In [1]:
import pandas as pd
import os
import glob

# 1. 自動尋找檔案（避免檔名手打錯誤）
# 搜尋包含 "1.xlsx" 的檔案
target_files = glob.glob("*1.xlsx*")

if not target_files:
    print("錯誤：找不到檔案！請確認檔案已上傳至 Colab 左側檔案區。")
else:
    file_path = target_files[0]
    print(f"偵測到檔案：{file_path}")

    # 根據副檔名讀取
    if file_path.endswith('.csv'):
        df = pd.read_csv(file_path)
    else:
        # 如果是 xlsx 格式，Colab 可能需要安裝 openpyxl
        df = pd.read_excel(file_path)

    # 2. 處理日期與新增 File 欄位
    # 根據你的資料，日期欄位名稱是 '年月日'
    date_col = '年月日'
    df[date_col] = pd.to_datetime(df[date_col])

    # 依照要求格式設定：OptionsDaily_YYYY_MM_DD.csv
    df['File'] = df[date_col].dt.strftime('OptionsDaily_%Y_%m_%d.csv')

    # 3. 調整欄位順序：把 File 移到 Date (年月日) 後面
    # 同時配合 Path_教學_0305 的格式將 '年月日' 改名為 'Date'
    df.rename(columns={'年月日': 'Date'}, inplace=True)

    cols = list(df.columns)
    # 將 'File' 移到第二欄
    cols.insert(1, cols.pop(cols.index('File')))
    df = df[cols]

    # 4. 輸出結果
    print("\n處理後前五筆資料：")
    print(df.head())

    # 儲存為 CSV
    output_name = 'Path_教學_Result.csv'
    df.to_csv(output_name, index=False, encoding='utf-8-sig')
    print(f"\n成功！請在左側檔案區下載：{output_name}")

偵測到檔案：1.xlsx

處理後前五筆資料：
        Date                         File    收盤價(元)
0 2023-01-03  OptionsDaily_2023_01_03.csv  14224.12
1 2023-01-04  OptionsDaily_2023_01_04.csv  14199.13
2 2023-01-05  OptionsDaily_2023_01_05.csv  14301.05
3 2023-01-06  OptionsDaily_2023_01_06.csv  14373.34
4 2023-01-09  OptionsDaily_2023_01_09.csv  14752.21

成功！請在左側檔案區下載：Path_教學_Result.csv


In [3]:
import pandas as pd

# 1. 讀取檔案
df_result = pd.read_csv('Path_教學_Result.csv')
df_expiry = pd.read_csv('資料來源3.csv')

# 2. 前處理：轉換日期格式
df_result['Date'] = pd.to_datetime(df_result['Date'])
df_expiry['最後結算日'] = pd.to_datetime(df_expiry['最後結算日'])

# 3. 篩選「月選擇權」：契約月份不包含 'W'
df_monthly = df_expiry[~df_expiry['契約月份'].str.contains('W', na=False)].copy()

# 確保結算日由小到大排序，方便後續尋找最近的到期日
df_monthly = df_monthly.sort_values('最後結算日').reset_index(drop=True)

def find_nearest_monthly_contract(trade_date):
    """
    針對交易日，尋找最近的一個月選擇權契約
    條件：最後結算日 - 交易日 > 1 (即至少剩餘 2 天)
    """
    # 篩選出結算日在 交易日 + 1天 之後的所有月契約
    valid_contracts = df_monthly[df_monthly['最後結算日'] > (trade_date + pd.Timedelta(days=1))]

    if not valid_contracts.empty:
        # 取最靠近的那一筆
        nearest = valid_contracts.iloc[0]
        contract_name = nearest['契約月份']
        expiry_date = nearest['最後結算日']
        # 計算 Maturity (剩餘天數)
        maturity = (expiry_date - trade_date).days
        return pd.Series([contract_name, expiry_date, maturity])
    else:
        return pd.Series([None, None, None])

# 4. 套用函式並新增欄位
df_result[['Contract', 'ContractExpiryDate', 'Maturity']] = df_result['Date'].apply(find_nearest_monthly_contract)

# 5. 格式化輸出日期 (選用，若希望儲存成 CSV 時日期格式較漂亮)
df_result['ContractExpiryDate'] = df_result['ContractExpiryDate'].dt.strftime('%Y/%m/%d')

# 顯示結果前 10 筆
print("處理完成後的結果預覽：")
print(df_result.head(10))

# 6. 儲存結果
df_result.to_csv('Path_教學_Processed_Result.csv', index=False, encoding='utf-8-sig')
print("\n已成功匯出檔案：Path_教學_Processed_Result.csv")

處理完成後的結果預覽：
        Date                         File    收盤價(元) Contract  \
0 2023-01-03  OptionsDaily_2023_01_03.csv  14224.12   202301   
1 2023-01-04  OptionsDaily_2023_01_04.csv  14199.13   202301   
2 2023-01-05  OptionsDaily_2023_01_05.csv  14301.05   202301   
3 2023-01-06  OptionsDaily_2023_01_06.csv  14373.34   202301   
4 2023-01-09  OptionsDaily_2023_01_09.csv  14752.21   202301   
5 2023-01-10  OptionsDaily_2023_01_10.csv  14802.96   202301   
6 2023-01-11  OptionsDaily_2023_01_11.csv  14751.44   202301   
7 2023-01-12  OptionsDaily_2023_01_12.csv  14731.64   202301   
8 2023-01-13  OptionsDaily_2023_01_13.csv  14824.13   202301   
9 2023-01-16  OptionsDaily_2023_01_16.csv  14927.01   202301   

  ContractExpiryDate  Maturity  
0         2023/01/30        27  
1         2023/01/30        26  
2         2023/01/30        25  
3         2023/01/30        24  
4         2023/01/30        21  
5         2023/01/30        20  
6         2023/01/30        19  
7         2023/01/3

In [4]:
import pandas as pd

# 1. 讀取資料
file_path = 'Path_教學_Processed_Result.csv'
df = pd.read_csv(file_path)

# 轉換日期格式以利對照
df['Date'] = pd.to_datetime(df['Date'])

# 2. 定義 2023 年台灣銀行利率變動歷史 (一年期定期儲蓄存款 - 機動利率)
# 數據來源：台灣銀行歷史利率查詢
rate_data = {
    'Effective_Date': ['2022-12-21', '2023-03-27', '2024-03-25'],
    'Rate': [1.465, 1.590, 1.715]  # 單位為 %
}
rates_df = pd.DataFrame(rate_data)
rates_df['Effective_Date'] = pd.to_datetime(rates_df['Effective_Date'])

# 3. 透過 merge_asof 進行「對照最近日期」的合併
# 這種方法會自動幫你找「小於或等於交易日」的最接近利率生效日
df = df.sort_values('Date')
rates_df = rates_df.sort_values('Effective_Date')

df = pd.merge_asof(df, rates_df,
                   left_on='Date',
                   right_on='Effective_Date',
                   direction='backward')

# 4. 整理欄位
# 將 Rf 轉換為小數形式 (例如 1.59% -> 0.0159)，這對後續計算 BS 模型較方便
df['Rf'] = df['Rate'] / 100

# 移除輔助用的 Effective_Date 與 Rate 欄位
df = df.drop(columns=['Effective_Date', 'Rate'])

# 5. 儲存並檢查結果
output_file = 'Path_教學_Processed_Result_with_Rf.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print("前 5 筆處理結果：")
print(df.head())
print(f"\n檔案已成功儲存至: {output_file}")

前 5 筆處理結果：
        Date                         File    收盤價(元)  Contract  \
0 2023-01-03  OptionsDaily_2023_01_03.csv  14224.12    202301   
1 2023-01-04  OptionsDaily_2023_01_04.csv  14199.13    202301   
2 2023-01-05  OptionsDaily_2023_01_05.csv  14301.05    202301   
3 2023-01-06  OptionsDaily_2023_01_06.csv  14373.34    202301   
4 2023-01-09  OptionsDaily_2023_01_09.csv  14752.21    202301   

  ContractExpiryDate  Maturity       Rf  
0         2023/01/30        27  0.01465  
1         2023/01/30        26  0.01465  
2         2023/01/30        25  0.01465  
3         2023/01/30        24  0.01465  
4         2023/01/30        21  0.01465  

檔案已成功儲存至: Path_教學_Processed_Result_with_Rf.csv
